<a href="https://colab.research.google.com/github/maqueda-09/trabajos4to/blob/main/BitcoinRNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RNN

In [ ]:
import pandas as pd

# Cargo el archivo CSV de los datos de Bitcoin. El separador es un punto y coma (';').
_df = pd.read_csv("/content/Bitcoin_1_1_2024-6_9_2024_historical_data_coinmarketcap.csv", delimiter=';')
# Los datos vienen desordenados, así que los invierto para que queden del más antiguo al más reciente.
_df = _df.sort_index(ascending=False)
# Para ver cómo se ve al inicio.
_df.head()

# --- Preprocesamiento de Datos ---
# Selecciono solo las columnas que me interesan.
# Las columnas del archivo original son:
# 'timeOpen', 'timeClose', 'timeHigh', 'timeLow', 'name', 'open', 'high',
# 'low', 'close', 'volume', 'marketCap', 'timestamp'
df = _df[['timeOpen', 'open', 'high', 'low', 'close']]
# Tomo solo los valores de 'close' (precio de cierre), que es lo que quiero predecir.
dates = df[['close']].values
# Uso el MinMaxScaler para escalar los datos. Esto los pone entre 0 y 1, que es bueno para la red neuronal.
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(dates)

# Defino el tamaño de la ventana de tiempo. Usaré 60 días para predecir el día 61.
window_size = 60
import numpy as np

# Función para crear las secuencias de datos. Esto es clave para las series de tiempo.
def create_sequences(data, window_size):
    sequences = []
    labels = []
    # Recorro los datos para crear ventanas.
    for i in range(len(data) - window_size):
        # La secuencia son los 60 días anteriores.
        sequences.append(data[i:i+window_size])
        # La etiqueta (lo que quiero predecir) es el precio del día siguiente.
        # El precio es la primera columna (columna 0), aunque aquí solo tengo una.
        labels.append(data[i + window_size, 0])
    return np.array(sequences), np.array(labels)

# Creo las secuencias. X son los datos de entrada (las ventanas), y 'y' son las salidas (las etiquetas).
X, y = create_sequences(scaled_data, window_size)

# Divido los datos en entrenamiento y prueba. Usaré el 80% para entrenar.
split = int(len(X) * 0.8)
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

# --- Construcción del Modelo RNN (SimpleRNN) ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, SimpleRNN

# Inicializo el modelo.
model = Sequential()
# Agrego la primera capa SimpleRNN. 'units' son las neuronas.
# 'return_sequences=True' es porque va a otra capa RNN después.
# 'input_shape' es (window_size, número de características), aquí es (60, 1).
model.add(SimpleRNN(units=120, return_sequences=True, input_shape=(window_size, X_train.shape[2])))
# Agrego la segunda capa SimpleRNN. 'return_sequences=False' porque después va una capa 'Dense'.
model.add(SimpleRNN(units=60, return_sequences=False))
# Capa 'Dense' normal con 30 neuronas.
model.add(Dense(units=30))
# Capa de salida con una sola neurona para predecir un solo valor (el precio).
model.add(Dense(units=1))

# --- Compilación y Entrenamiento ---
from tensorflow.keras.optimizers import Adam
# Defino un 'learning rate' para el optimizador Adam.
learning_rate = 0.001
adam_optimizer = Adam(learning_rate=learning_rate)

# Compilo el modelo. Uso 'mean_squared_error' (MSE) como 'loss' porque es un problema de regresión.
model.compile(optimizer=adam_optimizer, loss='mean_squared_error')

# Entreno el modelo. 'batch_size=1' significa que actualiza los pesos después de cada muestra (es lento pero a veces mejor).
# 'epochs=10' significa que pasa por todo el conjunto de entrenamiento 10 veces.
model.fit(X_train, y_train, batch_size=1, epochs=10)

# --- Evaluación y Desescalado ---
# Hago las predicciones con los datos de prueba.
predictions = model.predict(X_test)

# Para ver el resultado en dólares, necesito "desescalar" las predicciones.
# El 'MinMaxScaler' necesita 4 columnas (por como escalé los datos originales: timeOpen, open, high, low, close).
# Concateno las predicciones (que son la primera columna) con 3 columnas de ceros. Luego tomo solo la primera columna.
predictions = scaler.inverse_transform(np.concatenate((predictions, np.zeros((predictions.shape[0], 3))), axis=1))[:,0]
# Hago lo mismo con los valores reales de prueba para poder compararlos.
y_test = scaler.inverse_transform(np.concatenate((y_test.reshape(-1, 1), np.zeros((y_test.shape[0], 3))), axis=1))[:,0]

# Calculo las métricas para ver qué tan bueno fue el modelo.
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Error Absoluto Medio (MAE). Mide la diferencia promedio entre las predicciones y los valores reales.
mae = mean_absolute_error(y_test, predictions)
# Raíz del Error Cuadrático Medio (RMSE). Es más sensible a errores grandes.
rmse = np.sqrt(mean_squared_error(y_test, predictions))

# Muestro los resultados.
print(f"MAE: {mae}")
print(f"RMSE: {rmse}")

# --- Gráfico de Predicciones vs. Valores Reales ---
import matplotlib.pyplot as plt
import numpy as np

# Tomo la parte de los datos reales que se usó para la prueba.
valid = df[split:]
# Reseteo el índice para que no haya problemas al añadir la columna.
valid = valid.reset_index(drop=True)
# Creo una columna de 'Predictions' e inicialmente la lleno con 'NaN'.
valid['Predictions'] = np.nan

# Pongo las predicciones en la columna 'Predictions', empezando después de la 'window_size'.
# Esto es porque las primeras 'window_size' filas del conjunto de prueba no tienen predicción.
valid.loc[window_size:, 'Predictions'] = predictions
# Formateo las fechas para que se vean bien en el eje X del gráfico.
dates_valid = pd.to_datetime(valid['timeOpen']).apply(lambda x: x.strftime('%Y-%m-%d')).tolist()

# Creo el gráfico.
plt.figure(figsize=(16,8))
plt.title('Modelo RNN para Predicción de Bitcoin')
plt.xlabel('Fecha')
plt.ylabel('Precio de Bitcoin (USD)')
# Grafico el precio de cierre real y las predicciones.
plt.plot(dates_valid, valid[['close', 'Predictions']])
plt.legend(['Valor Real', 'Predicciones'], loc='lower right')
plt.xticks(rotation=90) # Roto las etiquetas de fecha para que no se superpongan.
plt.show()


# --- Predicción de Días Futuros ---
# Generar secuencias para los siguientes días
future_sequences = []
# La última secuencia real que tengo, que usaré para empezar a predecir el futuro.
last_sequence = X[-1]

days = 10 # Voy a predecir los próximos 10 días.
for _ in range(days):
    # Uso el modelo para predecir el siguiente valor.
    next_value = model.predict(np.array([last_sequence]))[0, 0]

    # **Actualizo la secuencia**: Muevo la ventana un día. Elimino el primer valor (el más viejo)
    # y añado la nueva predicción al final.
    last_sequence = np.concatenate((last_sequence[1:], [[next_value]]), axis=0)

    # Guardo la nueva secuencia.
    future_sequences.append(last_sequence)

# Convierto las secuencias futuras a un array de numpy.
future_sequences = np.array(future_sequences)
# Hago un 'reshape' para que tenga el formato que espera la RNN.
future_sequences = np.reshape(future_sequences, (future_sequences.shape[0], future_sequences.shape[1], 1))

# Hago las predicciones para todos los días futuros.
future_predictions = model.predict(future_sequences)

# Desescalo las predicciones futuras a dólares, igual que antes.
future_predictions = scaler.inverse_transform(np.concatenate((future_predictions, np.zeros((future_predictions.shape[0], 3))), axis=1))[:,0]

# Calculo las fechas futuras para el eje X.
# Tomo la última fecha del set de datos real.
last_date = df['timeOpen'].iloc[-1]
# Creo el rango de fechas futuras (excluyo el primer día porque ya lo predije, creo).
future_dates = pd.date_range(start=last_date, periods=days)[1:]
future_dates = future_dates.strftime('%Y-%m-%d').tolist()

# --- Gráfico de Predicciones Futuras ---
plt.figure(figsize=(16,8))
plt.title('Predicciones de los siguientes días')
plt.xlabel('Fecha')
plt.ylabel('Precio de Bitcoin (USD)')
# Grafico los datos de prueba con las predicciones.
plt.plot(dates_valid, valid[['close', 'Predictions']], label=['real', 'Predicciones'])
# Grafico las predicciones futuras (también excluyo el primer día).
plt.plot(future_dates, future_predictions[:-1], label='Predicciones')
plt.legend()
plt.xticks(rotation=90)
plt.show()